#### 共同骨架：A2C 的四个组成部分

1. 采集：`n_steps=32` 的短 roolout
```
n_steps=32, gamma=0.99, gae_lambda=0.95
```
A2C 每采 32 步就更新一次，每个并行环境各自独立采 32 步。对比：
- 第五章 REINFORCE：1 个回合（~200步）才更新一次
- 第一章 PPO：2048 步采一次，更新 10 个 epoch
> 更新频繁、每次数据少，所以吞吐快但样本效率低。

2. GAE(λ) 算优势
`gae_lambda=0.95` 让 Critic 承担大部分偏差消除工作。A2C 用的是同一个 Actor-Critic 网络（SB3 默认共享或分离两层 MLP），Critic 输出 `V(s)`，优势 `A = GAE(V, r)`。


与第五章 REINFORCE 比较：
|     | 第五章 REINFORCE  | 第六章 A2C                             |
| --- | -------------- | ----------------------------------- |
| 基线  | 学出来的 `V(s)`    | 学出来的 `V(s)`                         |
| 权重  | `G_t − V(s_t)` | `GAE` 优势（偏差-方差最优折中）                 |
| 归一化 | **无**          | **无**（`advantage_normalization` 未开） |

第六章新增的是 λ-GAE + 高频更新 + 连续动作分布。

3. 损失组合
```
policy_loss = -(log_prob * advantage).mean()
value_loss  = MSE(V(s), returns)
loss        = policy_loss + vf_coef * value_loss - ent_coef * entropy
```
`vf_coef=0.5`（Critic 权重减半，防止价值损失压过策略）、`max_grad_norm=0.5`（梯度裁剪）。

`ent_coef=0.0` 值得注意：连续控制里探索不靠熵奖励，而靠 Actor输出的高斯分布标准差 `std`。Pendulum 训练结束时 `std=0.915`，说明分布没坍缩——这是连续控制探索的主要机制。

4. 确定性评估

两个脚本评估都用 `model.predict(obs, deterministic=True)`，即取高斯分布的均值而非采样。所以评估方差反映的是初始状态差异，不是探索噪声——这解释了 BipedalWalker 为何 `std=62.6` 却大多数回合稳定在 260。

#### 1 维动作 + VecNormalize
状态/动作：
- 状态 3 维 `[cosθ, sinθ, θ̇]`，动作 1 维连续扭矩 `[-2, 2]`
- 回合固定 200 步，奖励每步 `-(θ² + 0.1θ̇² + 0.001τ²)`
关键设计：VecNormalize（`:174-175`）
```
vec_env = DummyVecEnv([make_env(seed, i) for i in range(8)])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
```
> 为什么 Pendulum 必须归一化：
- Pendulum 的 `θ̇`（角速度）是无界的，速度一大就是几十上百，直接喂进网络会让激活值爆炸

#### 4 维动作 + 无归一化 + 检查点
状态/动作：
- 状态 24 维 `[10 个 lidar, 4 个关节角, 4 个关节角速度, 速度 x, 速度 y]`
- 动作 4 维 `[左右髋扭矩, 左右膝扭矩]`，各 `[-1, 1]`
- 回合上限 1600 步，摔倒即终止

三处与 Pendulum 的设计差异:
① 不用 VecNormalize（`:174` 只有 DummyVecEnv） 如上所述，观测本身已归一化。
② 显式指定网络结构（`:187`）
```
policy_kwargs=dict(net_arch=[128, 128])
```
Pendulum 用 SB3 默认（Pendulum 默认更小）。BipedalWalker 需要显式指定，因为默认结构对 24 维输入 + 多关节协调不够。
③ 三阶段检查点（`:196-198`）
```
if args.total_timesteps >= 1_000_000:
    checkpoint_steps = [500_000, 1_000_000, 2_000_000]
```
只在长训练时启用，用于回答"策略在哪一步开始崩溃/收敛"。

#### 并排对照

|              | Pendulum        | BipedalWalker            |
| ------------ | --------------- | ------------------------ |
| 状态 / 动作维     | 3 / **1**       | 24 / **4**               |
| 并行环境         | 8               | 16                       |
| VecNormalize | **有**（观测+奖励）    | 无                        |
| 网络结构         | SB3 默认          | 显式 `[128,128]`           |
| 熵系数          | 0.0（靠高斯 std 探索） | 0.0                      |
| 参考分数         | -800（`:119`）    | 300 solved（`:118`）       |
| 评估方式         | 手动循环（需加载归一化）    | `evaluate_policy` + 手动双轮 |
| 检查点          | 无               | 500k/1M/2M               |
|300k/3M 步结果	|-771.0（达标）	|232.1（未达标）|